## 0. Configuración de rutas

In [2]:
import os

# El notebook vive en notebooks/ — subimos un nivel para llegar a la raiz del proyecto
PROJECT_PATH = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATOS_PATH   = os.path.join(PROJECT_PATH, "datos")

PROC_CC = os.path.join(DATOS_PATH, "processed", "cc_news")
PROC_MIND = os.path.join(DATOS_PATH, "processed", "mind_large")
TFIDF_CC = os.path.join(DATOS_PATH, "processed", "tfidf_cc")
TFIDF_MIND = os.path.join(DATOS_PATH, "processed", "tfidf_mind")
INDEX_PATH = os.path.join(DATOS_PATH, "processed", "inverted_index")
MODELS_PATH = os.path.join(PROJECT_PATH, "models")

for path in [TFIDF_CC, TFIDF_MIND, INDEX_PATH, MODELS_PATH]:
    os.makedirs(path, exist_ok=True)

print(f"PROJECT_PATH → {PROJECT_PATH}")
print("Rutas configuradas.")

PROJECT_PATH → /Users/milenafer/Desktop/datos_masivos/Proyecto_Datos_Masivos
Rutas configuradas.


## 1. Dependencias

In [3]:
# Ejecuta solo la primera vez
# !pip install pyspark nltk

In [4]:
import ssl
import nltk

ssl._create_default_https_context = ssl._create_unverified_context

nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

STOPWORDS_EN = set(stopwords.words("english"))
print(f"Stopwords cargadas: {len(STOPWORDS_EN)}")

Stopwords cargadas: 198


## 2. Inicialización de PySpark

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StringType

spark = (
    SparkSession.builder
    .appName("Preprocesamiento-MapReduce")
    .master("local[*]")
    .config("spark.driver.memory", "8g")
    .config("spark.driver.maxResultSize", "4g")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"PySpark {spark.version} listo.")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/20 18:29:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


PySpark 3.5.1 listo.


## 3. Carga de datos (salida de Fase 1)

In [6]:
cc_df = spark.read.parquet(PROC_CC)
mind_df = spark.read.parquet(PROC_MIND)

print(f"CC-News:    {cc_df.count():>8,} documentos")
print(f"MIND Large: {mind_df.count():>8,} documentos")

# Estandarizamos esquema: (doc_id, text, label)
# CC-News no tiene etiqueta -> None
cc_std = (
    cc_df
    .select(
        F.col("doc_id").cast("string"),
        F.col("text"),
        F.lit(None).cast("string").alias("label")))

# MIND tiene etiqueta de categoria
mind_std = (
    mind_df
    .select(
        F.col("news_id").alias("doc_id"),
        F.col("text"),
        F.col("category").alias("label")))

cc_std.show(2, truncate=80)
mind_std.show(2, truncate=80)

CC-News:     703,488 documentos
MIND Large:  173,550 documentos


+------+--------------------------------------------------------------------------------+-----+
|doc_id|                                                                            text|label|
+------+--------------------------------------------------------------------------------+-----+
| 31160|WannaCry (using the purloined exploit kit ETERNALBLUE) was paused, for now. H...| NULL|
| 65943|6 people take part in rare 3-way kidney transplant This three-way transplant ...| NULL|
+------+--------------------------------------------------------------------------------+-----+
only showing top 2 rows

+-------+--------------------------------------------------------------------------------+-----+
| doc_id|                                                                            text|label|
+-------+--------------------------------------------------------------------------------+-----+
|N104270|Kentucky Meat Shower? Yes, meat fell from the sky more than 140 years ago One...| news|
|N100124|Ho

## 4. MAP + COMBINER — Tokenización nativa JVM

Usamos `RegexTokenizer` y `StopWordsRemover` de MLlib (JVM puro) en lugar de UDFs Python.

Esto es 10-20x más rápido porque no hay serialización Python↔JVM por fila.

In [7]:
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover

# MAP: tokenizacion nativa JVM (sin UDF Python)
tokenizer = RegexTokenizer(
    inputCol="text",
    outputCol="tokens_raw",
    pattern="[^a-z]+",
    minTokenLength=3,
    toLowercase=True)

# COMBINER: eliminamos stopwords nativa JVM (sin UDF Python)
remover = StopWordsRemover(
    inputCol="tokens_raw",
    outputCol="tokens",
    stopWords=list(STOPWORDS_EN)
)

def limpiar(df):
    df_tok = tokenizer.transform(df)
    df_clean = remover.transform(df_tok).drop("tokens_raw")
    return df_clean.filter(F.size(F.col("tokens")) >= 5)

cc_limpio   = limpiar(cc_std)
mind_limpio = limpiar(mind_std)

print("Tokenización y limpieza definidas (nativas JVM — sin UDF Python).")
print("Ejemplo MIND:")
mind_limpio.select("doc_id", "tokens").limit(2).show(truncate=80)

Tokenización y limpieza definidas (nativas JVM — sin UDF Python).
Ejemplo MIND:
+-------+--------------------------------------------------------------------------------+
| doc_id|                                                                          tokens|
+-------+--------------------------------------------------------------------------------+
|N104270|[kentucky, meat, shower, yes, meat, fell, sky, years, ago, one, march, day, h...|
|N100124|[house, condemns, trump, syria, withdrawal, house, condemned, president, dona...|
+-------+--------------------------------------------------------------------------------+



26/05/20 18:29:48 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [8]:
from pyspark.ml.feature import HashingTF, IDF

NUM_FEATURES = 65536  # 2^16

hashing_tf = HashingTF(inputCol="tokens", outputCol="tf_vector", numFeatures=NUM_FEATURES)
idf        = IDF(inputCol="tf_vector", outputCol="tfidf_vector", minDocFreq=3)

# Solo calculamos TF-IDF para MIND (tiene etiquetas -> Fase 4)
# CC-News va directo a Fase 3 con tokens — MinHash no necesita TF-IDF
print("Calculando TF para MIND...")
tf_mind = hashing_tf.transform(mind_limpio)

print("Ajustando IDF para MIND (shuffle global)...")
idf_model_mind = idf.fit(tf_mind)
mind_tfidf = idf_model_mind.transform(tf_mind).select("doc_id", "label", "tokens", "tfidf_vector")

print("TF-IDF calculado.")
mind_tfidf.select("doc_id", "tfidf_vector").limit(2).show(truncate=80)

Calculando TF para MIND...
Ajustando IDF para MIND (shuffle global)...


TF-IDF calculado.
+-------+--------------------------------------------------------------------------------+
| doc_id|                                                                    tfidf_vector|
+-------+--------------------------------------------------------------------------------+
|N104270|(65536,[1232,3539,6964,10570,13003,20321,21823,21864,26498,27973,30432,31802,...|
|N100124|(65536,[5321,15148,22106,24193,34292,34909,37833,37936,51471,53110,55232,5552...|
+-------+--------------------------------------------------------------------------------+



26/05/20 18:29:59 WARN DAGScheduler: Broadcasting large task binary with size 1077.5 KiB


## 7. Guardado de vectores TF-IDF

In [9]:
import os

# 1. Guardar TF-IDF de MIND primero (173k docs — rápido)
print("Guardando TF-IDF MIND...")
(
    mind_tfidf
    .select("doc_id", "label", "tfidf_vector")
    .repartition(4)
    .write.mode("overwrite")
    .parquet(TFIDF_MIND)
)
print("TF-IDF MIND guardado.")

# Guardar modelo IDF
idf_model_mind.save(os.path.join(MODELS_PATH, "idf_model_mind"))
print("Modelo IDF guardado.")

# 2. CC-News: muestra de 100k docs (suficiente para demostrar MinHash LSH)
#    700k docs en local tardan >30 min; 100k tarda ~3 min y cubre el algoritmo
print("Guardando muestra de tokens CC-News (100 000 docs)...")
(
    cc_limpio
    .limit(100_000)
    .select("doc_id", "tokens")
    .repartition(4)
    .write.mode("overwrite")
    .parquet(os.path.join(DATOS_PATH, "processed", "tokens_cc"))
)
print("Tokens CC-News guardados (muestra 100k). Fase 2 completa.")

Guardando TF-IDF MIND...


26/05/20 18:30:00 WARN DAGScheduler: Broadcasting large task binary with size 1083.8 KiB


TF-IDF MIND guardado.


26/05/20 18:30:10 WARN TaskSetManager: Stage 18 contains a task of very large size (1052 KiB). The maximum recommended task size is 1000 KiB.


Modelo IDF guardado.
Guardando muestra de tokens CC-News (100 000 docs)...


Tokens CC-News guardados (muestra 100k). Fase 2 completa.


In [10]:
spark.stop()
print("SparkSession cerrada. Fase 2 completada.")
print("Siguiente paso: 03_lsh_dedup.ipynb")

SparkSession cerrada. Fase 2 completada.
Siguiente paso: 03_lsh_dedup.ipynb
